In [1]:
%pip install nltk pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import pandas as pd

df = pd.read_csv("../data/comments_raw.csv")
labels = pd.read_csv("../data/labels_manual.csv")
df = df.merge(labels, on="id", how="left")

NEG = {"không", "ko", "k", "chẳng", "chả", "chưa"}

def tokenize(text):
    tokens, neg = [], False
    for t in re.findall(r"\w+|[.,!?;]", text.lower()):
        if t in ".,!?;":
            neg = False
        else:
            tokens.append("NOT_" + t if neg else t)
            if t in NEG:
                neg = True
    return tokens

df["tokens"] = df["text"].apply(tokenize)
print(df["label"].value_counts(dropna=False))

label
NaN    180
pos     10
neg     10
Name: count, dtype: int64


In [3]:
import nltk

def features(tokens):
    return {f"contains({w})": True for w in set(tokens)}

labeled = df[df["label"].notna()]
train_set = [(features(t), l) for t, l in zip(labeled["tokens"], labeled["label"])]

classifier = nltk.NaiveBayesClassifier.train(train_set)
classifier.show_most_informative_features(10)

for s in ["claude đỉnh thật", "gemini phế"]:
    p = classifier.prob_classify(features(tokenize(s)))
    print(s, "->", p.max(), round(p.prob(p.max()), 2))

Most Informative Features
          contains(luôn) = True              pos : neg    =      3.0 : 1.0
            contains(là) = True              pos : neg    =      3.0 : 1.0
           contains(tâm) = True              pos : neg    =      2.3 : 1.0
        contains(NOT_có) = True              neg : pos    =      1.7 : 1.0
           contains(chứ) = True              neg : pos    =      1.7 : 1.0
            contains(mà) = True              pos : neg    =      1.7 : 1.0
           contains(mới) = True              neg : pos    =      1.7 : 1.0
         contains(người) = True              pos : neg    =      1.7 : 1.0
          contains(nhất) = True              pos : neg    =      1.7 : 1.0
           contains(thứ) = True              pos : neg    =      1.7 : 1.0
claude đỉnh thật -> pos 0.94
gemini phế -> neg 0.75


In [4]:
THRESHOLD = 0.9
df["pred"] = df["label"]                    
df["conf"] = None                         
df["round"] = None                           
df.loc[df["label"].notna(), "round"] = 0   

r = 0
while True:
    r += 1
    known = df[df["pred"].notna()]
    classifier = nltk.NaiveBayesClassifier.train(
        [(features(t), l) for t, l in zip(known["tokens"], known["pred"])])
    added = 0
    for i in df.index[df["pred"].isna()]:
        p = classifier.prob_classify(features(df.at[i, "tokens"]))
        if p.prob(p.max()) >= THRESHOLD:
            df.at[i, "pred"] = p.max()
            df.at[i, "conf"] = round(p.prob(p.max()), 2)
            df.at[i, "round"] = r
            added += 1
    print(f"Vong {r}: them {added} cau, tong co nhan {df['pred'].notna().sum()}")
    if added == 0:
        break

print(df["pred"].value_counts(dropna=False))

Vong 1: them 84 cau, tong co nhan 104
Vong 2: them 62 cau, tong co nhan 166
Vong 3: them 8 cau, tong co nhan 174
Vong 4: them 5 cau, tong co nhan 179
Vong 5: them 0 cau, tong co nhan 179
pred
neg    115
pos     64
NaN     21
Name: count, dtype: int64


In [5]:
df["pred"] = df["pred"].fillna("unknown")
df[["id", "text", "label", "pred", "conf", "round"]].to_csv("results.csv", index=False, encoding="utf-8-sig")
print(df["pred"].value_counts())
print(pd.crosstab(df["round"].fillna("-"), df["pred"]))

pred
neg        115
pos         64
unknown     21
Name: count, dtype: int64
pred   neg  pos  unknown
round                   
0       10   10        0
1       47   37        0
2       53    9        0
3        1    7        0
4        4    1        0
-        0    0       21


In [6]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
cols = ["text", "pred", "conf", "round"]

display(df[df["round"] == 0].set_index("id")[cols])
display(df[df["round"].fillna(0) > 0].sort_values("pred").set_index("id")[cols])
display(df[df["pred"] == "unknown"].set_index("id")[cols])

,text,pred,conf,round
id,,,,
8,"Tâm sự với người thì sượng lắm, với chatgpt nó cứ thủ thỉ, cứ nhiệt tình tận tâm, mua gói pro mà nói chuyện thì nó cứ gọi là quên lối về luôn 🤣🤣",pos,None,0
16,thật sứ thì gemini rất đần,neg,None,0
18,"ở đó mà mơ đi, AI là thứ tuyệt vời nhất con người tạo ra chỉ sau chip bán dẫn r. Nó sẽ giúp con người tiến rất xa trong 100 năm tới nên dĩ nhiên nó là xu thế r kh sập được đâu mấy con giời anti AI ngu",pos,None,0
45,con chatgpt giờ nó có trích cả link nguồn luôn r =)) đỡ phải đi kiếm nguồn,pos,None,0
46,"Hồi đó nó mới ra mắt và mình cũng dùng, nhưng nó viết ngu vcl. Thành ra t nhờ nó gợi ý dàn bài cho xong tự làm theo cách của t",neg,None,0
56,"Xưng mày tao với nó, train nó đủ nhiều, deep talk với nó đủ nhiều thì mỗi lần tâm sự là nó an ủi tốt tới mức muốn gặp nó ngoài đời😓",pos,None,0
57,"tui hong xài claude, còn gemini chắc chưa nạp nên nó hong có trích cho 🥲 vẫn là thấy chatgpt tiện nhất tại nó lấy nguồn đoạn nào nó gắn vô đằng sau lun",pos,None,0
59,"T dùng nó vì nó disposeable, v th. Chứ nó nhựa vl, nghe xong tht sự muốn đổ nuớc vào máy chủ",neg,None,0
70,phải bỏ tiền ra mới biết con claude đỉnh cỡ nào,pos,None,0


,text,pred,conf,round
id,,,,
1,"lý do là sử dụng AI vô tội vạ, prompt thì viết như loz ấy ông",neg,0.99,2
2,"Chúng mày để ý kỹ là nó rõ đến từng vết nứt trên tường luôn ý, cảm giác không phải do ai tạo",neg,1.0,2
3,Nếu có source rồi thì dùng notebookLM á mom,neg,0.94,1
7,"Ko, tâm sự với nó t ko thấy ngượng vì t ko coi nó là một người để được lắng nghe. T cần được nói ra suy nghĩ và đọc lại suy nghĩ của mình dưới góc nhìn khách quan nên t nói chuyện với nó",neg,1.0,2
9,Ủa k phải thằng hỏi AI nó ngu quá hả,neg,0.99,2
12,Ko ai xài Dola hả? Thấy seeding vậy chứ thấy cũng ngoan với giỏi mà 🤡,neg,0.9,1
14,chưa baog tin vào AI thề ngu dcd,neg,0.91,4
15,"Thì vẫn phải check liên tục chứ, nên nhớ AI chỉ hỗ trợ, chủ yếu vẫn do tư duy làm bài của mình nữa e, k nên phụ thuộc nhé. Mình dùng AI chứ k phải AI dùng mình.",neg,1.0,1
17,2026 vẫn có người đưa bài cho AI làm rồi nộp cho giảng viên hả trời 😬,neg,0.92,1


,text,pred,conf,round
id,,,,
5,"Có repo xoá cái đó mà izi thôi bro, vấn đề là người hỏi câu này thì khả năng là 100% làm cái là bị nhận ra 🥀",unknown,None,None
6,Tụi AI đọc văn tụi nó là nhận ra liền kkk =))) đọc phải dịch ra ngôn ngữ của mình r viết lại k à,unknown,None,None
10,"T mà làm tiểu luận chỉ nhờ mấy con AI lên sườn bài th rồi còn đâu sẽ tự làm mấy cái thông tin của bài, lên gg kiếm đầy luôn á",unknown,None,None
30,Thật sự là nó thanh xuân vườn trường vcd,unknown,None,None
34,"Ủa thấy nó khuyên cx okela lắm á, vs lại cx có nhiều chuyện kể vs bạn hay vs gdinh cx hơi bất tiện 1 xíu",unknown,None,None
42,"cú tui đi, chỉ tui mấy câu prompt hay hay cho con AI của tui nó khôn ra i",unknown,None,None
64,nghe nói gemini viết truyện hay lắm mà,unknown,None,None
73,M không nói tiếng người sao AI hiểu,unknown,None,None
87,vl t cgi cũng kể với con chat gpt😫,unknown,None,None


### Quy tắc chấm

| Trường hợp | Ghi id vào |
|---|---|
| Câu khen mà máy gán `neg` | `sai` |
| Câu chê mà máy gán `pos` | `sai` |
| Câu trung tính (hỏi đáp, kể chuyện, không khen không chê) mà máy vẫn gán `pos`/`neg` | `tt` |
| Máy gán đúng | không ghi gì |

Câu vừa khen vừa chê (ví dụ 105: “tạo ảnh hơi ngu. Còn lại thì bá”): chấm theo ý chính của câu. Câu 105 nghiêng về khen, nên máy gán pos là đúng. Câu nào khen chê ngang nhau thì ghi vào tt.

Câu khen/chê **người dùng** chứ không phải AI: ghi vào tt. Các câu cùng kiểu được chấm thống nhất.

In [7]:
sai = [2, 4, 7, 12, 27, 29, 33, 36, 53, 60, 72, 75, 84, 88, 97, 106, 113, 118, 123, 124, 126, 143, 145, 156, 159, 161, 164, 172, 175, 191, 199]
tt = [1, 3, 9, 11, 15, 17, 21, 23, 24, 28, 32, 37, 38, 39, 41, 47, 48, 49, 50, 51, 55, 58, 61, 62, 65, 66, 67, 68, 69, 71, 74, 77, 78, 79, 80, 82, 83, 86, 89, 92, 93, 94, 96, 98, 99, 100, 101, 107, 108, 109, 112, 121, 128, 132, 134, 136, 141, 144, 147, 151, 152, 154, 157, 162, 163, 166, 167, 168, 170, 171, 173, 176, 178, 186, 193, 194, 195, 196, 198]

n = (df["round"].fillna(0) > 0).sum()
print("Tong:", n, "| Dung:", n - len(sai) - len(tt), "| Sai:", len(sai), "| Trung tinh:", len(tt))
print("Ti le dung:", round((n - len(sai) - len(tt)) / n * 100), "%")

Tong: 159 | Dung: 49 | Sai: 31 | Trung tinh: 79
Ti le dung: 31 %
